# 🎯 Project 5 - Number Guessing Game

## What it demonstrates
| Concept | Where used |
|---------|------------|
| `random` module | Secret number generation |
| while loop | Guess until correct |
| Exception handling | Non-integer input |
| Dataclasses | `GameConfig`, `AttemptRecord` |
| Statistics | Average guesses, best game, score |
| Binary search intuition | Optimal strategy demo |
| Enums | `Difficulty` level |

## Difficulty levels
| Level | Range | Max guesses | Hints |
|-------|-------|-------------|-------|
| Easy | 1–50 | 10 | Hot/cold |
| Medium | 1–100 | 7 | Higher/Lower |
| Hard | 1–200 | 6 | Higher/Lower only |

## Scoring
```
Base score  = 1000
Penalty     = 100 × (attempts - 1)
Time bonus  = max(0, 500 - seconds_taken × 10)
Final score = base - penalty + time_bonus
```

In [1]:
# ============================================================
#  PROJECT 5 — NUMBER GUESSING GAME
# ============================================================

import random, time
from dataclasses import dataclass, field
from typing import List, Optional
from enum import Enum

class Difficulty(Enum):
    EASY   = ("Easy",   1,   50,  10)
    MEDIUM = ("Medium", 1,  100,   7)
    HARD   = ("Hard",   1,  200,   6)

    def __init__(self, label, lo, hi, max_guesses):
        self.label       = label
        self.lo          = lo
        self.hi          = hi
        self.max_guesses = max_guesses

@dataclass
class AttemptRecord:
    game_num:    int
    secret:      int
    guesses:     List[int]
    won:         bool
    time_taken:  float
    difficulty:  Difficulty

    @property
    def num_guesses(self) -> int:
        return len(self.guesses)

    @property
    def score(self) -> int:
        if not self.won: return 0
        base       = 1000
        penalty    = 100 * (self.num_guesses - 1)
        time_bonus = max(0, int(500 - self.time_taken * 10))
        return max(0, base - penalty + time_bonus)

class GuessingGame:
    """Number guessing game with difficulty levels and statistics."""

    def __init__(self, difficulty: Difficulty = Difficulty.MEDIUM, player: str = "Player"):
        self.difficulty  = difficulty
        self.player      = player
        self.history: List[AttemptRecord] = []

    def _hint(self, secret: int, guess: int, attempt: int) -> str:
        diff  = abs(secret - guess)
        arrow = "⬆️ " if guess < secret else "⬇️ "

        if self.difficulty == Difficulty.EASY:
            if diff == 0:   temp = "🎯 Exactly right!"
            elif diff <= 3: temp = f"{arrow} 🔥 Burning hot!"
            elif diff <= 8: temp = f"{arrow} ♨️  Very warm!"
            elif diff <= 15: temp = f"{arrow} 🌡️  Getting warm..."
            else:            temp = f"{arrow} 🥶 Ice cold."
        else:
            if diff == 0:  temp = "🎯 Exactly right!"
            else:          temp = f"{arrow} Go {'higher' if guess < secret else 'lower'}!"

        return temp

    def play_one(self, simulated_guesses: Optional[List[int]] = None) -> AttemptRecord:
        """Play one game. Pass simulated_guesses for automated demo."""
        d      = self.difficulty
        secret = random.randint(d.lo, d.hi)
        guesses: List[int] = []
        won    = False
        start  = time.perf_counter()
        game_n = len(self.history) + 1

        print(f"\n  Game {game_n} │ {d.label} │ Range: {d.lo}–{d.hi} │ Max guesses: {d.max_guesses}")
        print(f"  {'─'*50}")

        for attempt in range(1, d.max_guesses + 1):
            remaining = d.max_guesses - attempt

            # Get guess — from simulation list or binary-search auto-play
            if simulated_guesses and attempt <= len(simulated_guesses):
                guess = simulated_guesses[attempt - 1]
            else:
                # Auto binary-search strategy
                lo_b = d.lo if not guesses else max(
                    g for g in guesses if g < secret) if any(g < secret for g in guesses) else d.lo
                hi_b = d.hi if not guesses else min(
                    g for g in guesses if g > secret) if any(g > secret for g in guesses) else d.hi
                guess = (lo_b + hi_b) // 2

            guesses.append(guess)
            hint = self._hint(secret, guess, attempt)
            print(f"  Attempt {attempt:2d}/{d.max_guesses} │ Guess: {guess:4d} │ {hint}"
                  + (f" │ {remaining} left" if remaining > 0 and guess != secret else ""))

            if guess == secret:
                won = True
                break

        elapsed = time.perf_counter() - start
        record  = AttemptRecord(game_n, secret, guesses, won, elapsed, d)
        self.history.append(record)

        print(f"  {'─'*50}")
        if won:
            print(f"  🎉 Correct! The number was {secret}. "
                  f"Guesses: {record.num_guesses} │ Score: {record.score}")
        else:
            print(f"  😔 Out of guesses! The number was {secret}.")

        return record

    def overall_stats(self):
        if not self.history:
            print("  No games played yet.")
            return
        wins  = [r for r in self.history if r.won]
        total = len(self.history)
        print(f"\n  {'═'*50}")
        print(f"  📊 OVERALL STATS — {self.player}")
        print(f"  {'═'*50}")
        print(f"  Games played : {total}")
        print(f"  Games won    : {len(wins)} ({len(wins)/total*100:.0f}%)")
        if wins:
            avg_g = sum(r.num_guesses for r in wins) / len(wins)
            best  = min(wins, key=lambda r: r.num_guesses)
            top_s = max(wins, key=lambda r: r.score)
            print(f"  Avg guesses  : {avg_g:.1f}")
            print(f"  Best game    : Game {best.game_num} ({best.num_guesses} guesses, secret={best.secret})")
            print(f"  High score   : {top_s.score} (Game {top_s.game_num})")
            print(f"  Total score  : {sum(r.score for r in self.history)}")
        print(f"  {'═'*50}")

print("✅ Number Guessing Game defined.")

✅ Number Guessing Game defined.


In [2]:
# ---- Demo — 3 automated games across difficulties ----

random.seed(7)

print("=" * 52)
print("      🎯  NUMBER GUESSING GAME")
print("=" * 52)

# Game 1 — Easy with some human-like guesses
g_easy = GuessingGame(Difficulty.EASY,   "Purvi")
g_easy.play_one(simulated_guesses=[25, 38, 42, 45])

# Game 2 — Medium auto binary-search
g_med = GuessingGame(Difficulty.MEDIUM,  "Purvi")
g_med.play_one()   # binary search auto-play
g_med.play_one()   # second game

# Game 3 — Hard auto binary-search
g_hard = GuessingGame(Difficulty.HARD,   "Purvi")
g_hard.play_one()

# Show stats for Medium (2 games)
g_med.overall_stats()

# Show binary-search strategy explanation
print("\n  💡 Optimal strategy: Binary Search")
print("     Always guess the midpoint of remaining range.")
print("     Worst case: ⌈log₂(range)⌉ guesses.")
print(f"     For 1–100: ⌈log₂(100)⌉ = 7 guesses max ✓")
print(f"     For 1–200: ⌈log₂(200)⌉ = 8 guesses max ✓")

      🎯  NUMBER GUESSING GAME

  Game 1 │ Easy │ Range: 1–50 │ Max guesses: 10
  ──────────────────────────────────────────────────
  Attempt  1/10 │ Guess:   25 │ ⬇️  ♨️  Very warm! │ 9 left
  Attempt  2/10 │ Guess:   38 │ ⬇️  🥶 Ice cold. │ 8 left
  Attempt  3/10 │ Guess:   42 │ ⬇️  🥶 Ice cold. │ 7 left
  Attempt  4/10 │ Guess:   45 │ ⬇️  🥶 Ice cold. │ 6 left
  Attempt  5/10 │ Guess:   13 │ ⬆️  ♨️  Very warm! │ 5 left
  Attempt  6/10 │ Guess:   19 │ ⬆️  🔥 Burning hot! │ 4 left
  Attempt  7/10 │ Guess:   22 │ ⬇️  🔥 Burning hot! │ 3 left
  Attempt  8/10 │ Guess:   20 │ ⬆️  🔥 Burning hot! │ 2 left
  Attempt  9/10 │ Guess:   21 │ 🎯 Exactly right!
  ──────────────────────────────────────────────────
  🎉 Correct! The number was 21. Guesses: 9 │ Score: 699

  Game 1 │ Medium │ Range: 1–100 │ Max guesses: 7
  ──────────────────────────────────────────────────
  Attempt  1/7 │ Guess:   50 │ ⬇️  Go lower! │ 6 left
  Attempt  2/7 │ Guess:   25 │ ⬇️  Go lower! │ 5 left
  Attempt  3/7 │ Guess:   1